# Bank Customer Churn Prediction

**Author:** Ashish Singh  
**Dataset:** [Bank Customer Churn - Kaggle](https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction)  
**Objective:** Build a machine-learning model that predicts whether a bank customer will **churn** (leave the bank) based on their demographics, account details, and banking behaviour.

---

## Business Context

Customer churn is one of the most critical metrics in the banking industry. Acquiring a new customer costs **5-7x more** than retaining an existing one.  
By accurately predicting which customers are likely to leave, the bank's marketing team can:

- **Proactively reach out** with personalised retention offers.  
- **Identify pain-points** in the product or service experience.  
- **Optimise marketing spend** by focusing on high-risk segments.

---

## Notebook Roadmap

| # | Section |
|---|---|
| 1 | Imports & Setup |
| 2 | Load Dataset |
| 3 | Data Overview & Data Dictionary |
| 4 | Missing Values & Data Cleaning |
| 5 | Exploratory Data Analysis (EDA) |
| 6 | Feature Engineering & Preprocessing |
| 7 | Model Training |
| 8 | Model Evaluation & Comparison |
| 9 | Hyperparameter Tuning |
| 10 | Feature Importance & SHAP Interpretability |
| 11 | Conclusion & Business Recommendations |
| 12 | Save Model |

---
## 1. Imports & Setup

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Modelling
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Interpretability
import shap

# Persistence
import joblib
import os

# Plot Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

---
## 2. Load Dataset

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

---
## 3. Data Overview & Data Dictionary

| Column | Description | Type |
|--------|-------------|------|
| `RowNumber` | Row index (irrelevant) | int |
| `CustomerId` | Unique customer ID (irrelevant) | int |
| `Surname` | Customer last name (irrelevant) | str |
| `CreditScore` | Credit score of the customer | int |
| `Geography` | Country - France, Spain, or Germany | str |
| `Gender` | Male or Female | str |
| `Age` | Customer age | int |
| `Tenure` | Years the customer has been with the bank | int |
| `Balance` | Account balance | float |
| `NumOfProducts` | Number of bank products used | int |
| `HasCrCard` | Has a credit card? (1 = Yes, 0 = No) | int |
| `IsActiveMember` | Is an active member? (1 = Yes, 0 = No) | int |
| `EstimatedSalary` | Estimated annual salary | float |
| **`Exited`** | **Target - Did the customer churn? (1 = Yes, 0 = No)** | **int** |

In [ ]:
# Statistical summary of numerical features
df.describe()

In [ ]:
# Data types and non-null counts
df.info()

---
## 4. Missing Values & Data Cleaning

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

### Dropping Irrelevant Columns

The following columns have **no predictive power** for churn:

- `RowNumber` - just a sequential index  
- `CustomerId` - a unique identifier, not a feature  
- `Surname` - a name carries no business-relevant signal for churn prediction

In [ ]:
# Drop columns that will not help in prediction
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
print(f'Shape after cleaning: {df.shape}')
df.head()

---
## 5. Exploratory Data Analysis (EDA)

Let's understand the data visually before modelling.

### 5.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
churn_counts = df['Exited'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Stayed (0)', 'Churned (1)'], churn_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['Stayed', 'Churned'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 13, 'fontweight': 'bold'})
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

churn_rate = churn_counts[1] / churn_counts.sum() * 100
print(f'\nChurn rate: {churn_rate:.1f}%  -->  The dataset is IMBALANCED (we will handle this during modelling).')

### 5.2 Distribution of Numerical Features

In [ ]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    # Plot separate histograms for stayed vs churned
    axes[i].hist(df[df['Exited']==0][col], bins=30, alpha=0.6, color='#2ecc71', label='Stayed', edgecolor='black')
    axes[i].hist(df[df['Exited']==1][col], bins=30, alpha=0.6, color='#e74c3c', label='Churned', edgecolor='black')
    axes[i].set_title(f'{col} Distribution by Churn', fontsize=12, fontweight='bold')
    axes[i].legend()

plt.suptitle('Numerical Feature Distributions - Churned vs Stayed', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Age & Balance - Boxplots by Churn Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create a temporary string column for plotting
plot_df = df.copy()
plot_df['Churn_Status'] = plot_df['Exited'].map({0: 'Stayed', 1: 'Churned'})

sns.boxplot(data=plot_df, x='Churn_Status', y='Age',
            palette=['#2ecc71', '#e74c3c'], order=['Stayed', 'Churned'], ax=axes[0])
axes[0].set_title('Age vs Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')

sns.boxplot(data=plot_df, x='Churn_Status', y='Balance',
            palette=['#2ecc71', '#e74c3c'], order=['Stayed', 'Churned'], ax=axes[1])
axes[1].set_title('Balance vs Churn', fontsize=14, fontweight='bold')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print('Insight: Older customers and customers with higher balances tend to churn more.')

### 5.4 Categorical Features - Geography & Gender vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create a string label column for the legend
plot_df = df.copy()
plot_df['Churn_Label'] = plot_df['Exited'].map({0: 'Stayed', 1: 'Churned'})

# Geography
sns.countplot(data=plot_df, x='Geography', hue='Churn_Label',
              palette=['#2ecc71', '#e74c3c'], hue_order=['Stayed', 'Churned'], ax=axes[0])
axes[0].set_title('Churn by Geography', fontsize=14, fontweight='bold')
axes[0].legend(title='Status')

# Gender
sns.countplot(data=plot_df, x='Gender', hue='Churn_Label',
              palette=['#2ecc71', '#e74c3c'], hue_order=['Stayed', 'Churned'], ax=axes[1])
axes[1].set_title('Churn by Gender', fontsize=14, fontweight='bold')
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

# Churn rates
print('\nChurn Rate by Geography:')
print(df.groupby('Geography')['Exited'].mean().apply(lambda x: f'{x*100:.1f}%'))
print('\nChurn Rate by Gender:')
print(df.groupby('Gender')['Exited'].mean().apply(lambda x: f'{x*100:.1f}%'))
print('\nInsight: Germany has the highest churn rate. Female customers churn more than males.')

### 5.5 Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))

corr = df.select_dtypes(include=[np.number]).corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Insight: Age has the strongest positive correlation with Exited (churn).\n'
      '   IsActiveMember has a negative correlation - active members churn less.')

### 5.6 Number of Products vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
churn_by_products = df.groupby('NumOfProducts')['Exited'].mean() * 100
churn_by_products.plot(kind='bar', color=['#2ecc71', '#27ae60', '#e67e22', '#e74c3c'],
                       edgecolor='black', ax=ax)
ax.set_title('Churn Rate by Number of Products', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Number of Products')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

for i, v in enumerate(churn_by_products.values):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Insight: Customers with 3 or 4 products have extremely high churn rates!\n'
      '   This may indicate over-selling or product-fit issues.')

---
## 6. Feature Engineering & Preprocessing

In [ ]:
# Encode Gender: Male -> 1, Female -> 0
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
print('Gender encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# One-Hot Encode Geography
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

print(f'\nShape after encoding: {df.shape}')
df.head()

---
## 7. Train / Test Split & Scaling

In [ ]:
# Separate features and target
X = df.drop(columns=['Exited'])
y = df['Exited']

print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Target distribution:\n{y.value_counts()}')

# Stratified split (preserves class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train churn rate: {y_train.mean()*100:.1f}%')
print(f'Test  churn rate: {y_test.mean()*100:.1f}%')

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nData split and scaled.')

---
## 8. Model Training

We train **four models** and compare their performance.  
Since the dataset is **imbalanced** (~20% churn), we use:
- `class_weight='balanced'` for sklearn models  
- `scale_pos_weight` for XGBoost  

This penalises the model more for misclassifying the minority class (churned customers).

In [ ]:
# Calculate class imbalance ratio for XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Class imbalance ratio (non-churn / churn): {scale_pos:.2f}')

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, scale_pos_weight=scale_pos,
        eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0
    )
}

# Train all models
results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_proba)
    }
    print(f'   {name}  ->  F1={results[name]["F1 Score"]:.4f}  |  ROC-AUC={results[name]["ROC-AUC"]:.4f}')

print('\nAll models trained!')

---
## 9. Model Evaluation & Comparison

In [ ]:
# Results comparison table
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('F1 Score', ascending=False)
results_df.style.background_gradient(cmap='RdYlGn', axis=0).format('{:.4f}')

In [ ]:
# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind='bar', ax=ax, edgecolor='black', width=0.8)
ax.set_title('Model Comparison - All Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices - All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))

colors_roc = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
for (name, model), color in zip(models.items(), colors_roc):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 10. Hyperparameter Tuning

We use **RandomizedSearchCV** on XGBoost to find optimal hyperparameters.

In [ ]:
# Identify best model by F1 score
best_model_name = results_df['F1 Score'].idxmax()
print(f'Best model (by F1): {best_model_name}')
print('Now tuning XGBoost hyperparameters with RandomizedSearchCV...\n')

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3],
}

xgb_tuned = XGBClassifier(
    scale_pos_weight=scale_pos, eval_metric='logloss',
    random_state=42, n_jobs=-1, verbosity=0
)

search = RandomizedSearchCV(
    xgb_tuned, param_distributions,
    n_iter=50, scoring='f1', cv=5, random_state=42, n_jobs=-1, verbose=1
)

search.fit(X_train_scaled, y_train)

print(f'\nBest F1 Score (CV): {search.best_score_:.4f}')
print(f'\nBest Hyperparameters:')
for param, val in search.best_params_.items():
    print(f'   {param}: {val}')

In [ ]:
# Evaluate tuned model on test set
best_model = search.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
y_proba_tuned = best_model.predict_proba(X_test_scaled)[:, 1]

print('=' * 50)
print('  TUNED XGBoost - Test Set Performance')
print('=' * 50)
print(classification_report(y_test, y_pred_tuned, target_names=['Stayed', 'Churned']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_tuned):.4f}')

# Compare before vs after tuning
print('\nTuning Improvement:')
print(f'   F1 (before): {results["XGBoost"]["F1 Score"]:.4f}')
print(f'   F1 (after):  {f1_score(y_test, y_pred_tuned):.4f}')

---
## 11. Feature Importance & SHAP Interpretability

Understanding **why** a model predicts churn is just as important as the prediction itself.  
This section helps the business team identify the key drivers of customer attrition.

In [ ]:
# Built-in feature importance (from tuned XGBoost)
feature_names = X.columns.tolist()
importances = best_model.feature_importances_
fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
fi.plot(kind='barh', color='#3498db', edgecolor='black')
plt.title('Feature Importance - Tuned XGBoost', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### SHAP Summary Plot

SHAP (SHapley Additive exPlanations) shows the **direction** and **magnitude** of each feature's impact on the prediction.

In [ ]:
# SHAP explainer
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

# Summary plot (beeswarm)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_names, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('How to read this plot:')
print('   - Each dot = one customer')
print('   - Red = high feature value, Blue = low feature value')
print('   - Dots to the RIGHT push prediction toward CHURN')
print('   - Dots to the LEFT push prediction toward STAY')

---
## 12. Conclusion & Business Recommendations

### Key Findings

1. **Age is the strongest predictor of churn.** Older customers (40+) are significantly more likely to leave.  
2. **Geography matters.** Customers in **Germany** have the highest churn rate (~32%), compared to France (~16%) and Spain (~17%).  
3. **Inactive members churn more.** Customers who are not active banking members are at higher risk.  
4. **Number of products is a double-edged sword.** Customers with 3-4 products almost always churn - suggesting possible over-selling.  
5. **Gender gap.** Female customers churn at a higher rate than male customers.  
6. **Balance paradox.** Customers with higher balances churn more - they may have more options at competing banks.

### Business Recommendations

| Recommendation | Target Segment | Expected Impact |
|---|---|---|
| Launch a **loyalty program** with personalised offers | Customers aged 40+ | Reduce churn in highest-risk age group |
| Investigate **Germany-specific issues** (service quality, fees, competition) | German customers | Address the country with 2x churn rate |
| Create **re-engagement campaigns** for inactive members | IsActiveMember = 0 | Convert passive customers to active ones |
| **Review product bundling** strategy | Customers with 3+ products | Prevent over-selling that leads to churn |
| Offer **competitive savings rates** to high-balance customers | Balance > 100K | Retain valuable customers being poached |

### Model Performance

The **tuned XGBoost model** achieved the best performance with balanced precision and recall, making it suitable for deployment in a production churn-prediction pipeline.

---
## 13. Save Model

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# Save the tuned model and the scaler
joblib.dump(best_model, 'models/best_churn_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print('Saved:')
print('   models/best_churn_model.joblib')
print('   models/scaler.joblib')
print('\nProject complete!')